# Glob residuation playground

What happens between *"a pattern arrives at the router"* and *"each mounted
backend gets its local pattern"* — the algebra in
`src/vfs/pattern_matching/glob.py`: `residuals` (l.178), `composed_pattern`
(l.160), `_canonical` (l.224), consumed by the router at `base.py:1960`.

Every `trace(...)` below re-runs the NFA frontier step by step **and asserts
its final frontier equals the real `residuals()`**, so the printouts cannot
drift from the code.

Mental model: the mount path is the input tape, each residual tuple is one
NFA state ("what's left of the pattern"), and `**` is the only component
that forks the frontier — it can consume a segment *and stay*, or stand
aside while the next component consumes.

In [6]:
from vfs.paths import Path
from vfs.pattern_matching import (
    composed_pattern,
    effective_pattern,
    filter_paths,
    glob_defect,
    render_residual,
    residuals,
)
from vfs.pattern_matching.glob import _canonical, _component_matches

In [7]:
def trace(pattern: str, mount: str) -> None:
    """Print the NFA frontier after each mount segment; assert against residuals()."""
    frontier = {tuple(_canonical(pattern).strip("/").split("/"))}
    print(f"pattern {pattern!r}  vs  mount {mount!r}")
    print(f"  start          : {sorted(frontier)}")
    for segment in (part for part in mount.split("/") if part):
        advanced = set()
        for candidate in frontier:
            if not candidate:
                continue  # pattern already fully consumed above the mount -> dies
            head, rest = candidate[0], candidate[1:]
            if head == "**":
                advanced.add(candidate)  # ** consumes the segment and stays
                if rest and rest[0] != "**" and _component_matches(rest[0], segment):
                    advanced.add(rest[1:])  # zero-match arm: the NEXT component consumed it
            elif _component_matches(head, segment):
                advanced.add(rest)
        frontier = advanced
        print(f"  after {segment!r:<9}: {sorted(frontier)}")
    result = residuals(pattern, Path(mount))
    assert frozenset(frontier) == result, "trace drifted from residuals()!"
    print(f"  residuals()    = {result}")
    if not result:
        print("                   # empty set: no residual survived -> router skips this mount")
    for c in sorted(result):
        if not c:
            print("                   # () : pattern matches the mount dir itself -- a parent-owned row,")
            print("                   #      dropped from child dispatch at base.py:1960")
        else:
            print(f"                   # {c} renders to local pattern {render_residual(c)!r}")
    print()

## 1. The three basic outcomes — live, dead, bind point

The running journey example: mount a backend at `/src/util` and throw three
patterns at it. Watch the frontier: a literal component consumes exactly its
segment; when the frontier empties the mount is **dead** (`base.py:1967`
skips it entirely); when a candidate empties on the *last* segment the
pattern matched the mount-point directory itself — a row stored in the
**parent** backend, so `base.py:1957` drops it from child dispatch.

In [8]:
trace("/src/**/*.py", "/src/util")   # live: backend runs /**/*.py locally
trace("/docs/**",     "/src/util")   # dead: 'docs' never consumes 'src'
trace("/src/*",       "/src/util")   # bind point: pattern matched the mount dir itself

pattern '/src/**/*.py'  vs  mount '/src/util'
  start          : [('src', '**', '*.py')]
  after 'src'    : [('**', '*.py')]
  after 'util'   : [('**', '*.py')]
  residuals()    = frozenset({('**', '*.py')})
                   # ('**', '*.py') renders to local pattern '/**/*.py'

pattern '/docs/**'  vs  mount '/src/util'
  start          : [('docs', '**')]
  after 'src'    : []
  after 'util'   : []
  residuals()    = frozenset()
                   # empty set: no residual survived -> router skips this mount

pattern '/src/*'  vs  mount '/src/util'
  start          : [('src', '*')]
  after 'src'    : [('*',)]
  after 'util'   : [()]
  residuals()    = frozenset({()})
                   # () : pattern matches the mount dir itself -- a parent-owned row,
                   #      dropped from child dispatch at base.py:1960



## 2. The `**` fork — one segment, two futures

When the head is `**` and the *next* component also matches the segment,
both futures are real: the `**` ate it, or the `**` matched zero segments
and the next component ate it. The frontier holds both, and dispatch honors
both residuals.

In [9]:
trace("/src/**/util/*.py", "/src/util")   # fork: is the mount's util THE util?

pattern '/src/**/util/*.py'  vs  mount '/src/util'
  start          : [('src', '**', 'util', '*.py')]
  after 'src'    : [('**', 'util', '*.py')]
  after 'util'   : [('**', 'util', '*.py'), ('*.py',)]
  residuals()    = frozenset({('*.py',), ('**', 'util', '*.py')})
                   # ('**', 'util', '*.py') renders to local pattern '/**/util/*.py'
                   # ('*.py',) renders to local pattern '/*.py'



In [15]:
# Chain more `**/literal` pairs over a mount that repeats the literal and the
# frontier keeps widening -- three live residuals from one pattern:
trace("**/a/**/b/*.txt", "/b")

pattern '**/a/**/b/*.txt'  vs  mount '/b'
  start          : [('**', 'a', '**', 'b', '*.txt')]
  after 'b'      : [('**', 'a', '**', 'b', '*.txt')]
  residuals()    = frozenset({('**', 'a', '**', 'b', '*.txt')})
                   # ('**', 'a', '**', 'b', '*.txt') renders to local pattern '/**/a/**/b/*.txt'



In [24]:
# The set also DEDUPES futures that converge on the same leftover:
# mount /a/b/b -- two different choice paths both leave ('**', 'c.txt').
trace("/a/**/b/**/c.txt", "/a/b/e")

pattern '/a/**/b/**/c.txt'  vs  mount '/a/b/e'
  start          : [('a', '**', 'b', '**', 'c.txt')]
  after 'a'      : [('**', 'b', '**', 'c.txt')]
  after 'b'      : [('**', 'b', '**', 'c.txt'), ('**', 'c.txt')]
  after 'e'      : [('**', 'b', '**', 'c.txt'), ('**', 'c.txt')]
  residuals()    = frozenset({('**', 'c.txt'), ('**', 'b', '**', 'c.txt')})
                   # ('**', 'b', '**', 'c.txt') renders to local pattern '/**/b/**/c.txt'
                   # ('**', 'c.txt') renders to local pattern '/**/c.txt'



## 3. Canonicalization, character classes, `?`, and defects

`_canonical` collapses adjacent `**` before the walk starts (`**/**` matches
exactly what `**` matches, and the extra component would starve the
zero-match arm — glob.py:224). Classes and `?` are ordinary one-segment
consumers via `_component_matches`. Malformed patterns never get this far:
`glob_defect` refuses them loudly upstream.

In [16]:
trace("/a/**/**/b", "/a")                          # canonical form is /a/**/b
trace("/data/v[0-9]/**/part-??.csv", "/data/v3")   # class consumes v3
trace("/data/v[0-9]/**/part-??.csv", "/data/vX")   # vX fails the class -> dead

pattern '/a/**/**/b'  vs  mount '/a'
  start          : [('a', '**', 'b')]
  after 'a'      : [('**', 'b')]
  residuals()    = frozenset({('**', 'b')})
                   # ('**', 'b') renders to local pattern '/**/b'

pattern '/data/v[0-9]/**/part-??.csv'  vs  mount '/data/v3'
  start          : [('data', 'v[0-9]', '**', 'part-??.csv')]
  after 'data'   : [('v[0-9]', '**', 'part-??.csv')]
  after 'v3'     : [('**', 'part-??.csv')]
  residuals()    = frozenset({('**', 'part-??.csv')})
                   # ('**', 'part-??.csv') renders to local pattern '/**/part-??.csv'

pattern '/data/v[0-9]/**/part-??.csv'  vs  mount '/data/vX'
  start          : [('data', 'v[0-9]', '**', 'part-??.csv')]
  after 'data'   : [('v[0-9]', '**', 'part-??.csv')]
  after 'vX'     : []
  residuals()    = frozenset()
                   # empty set: no residual survived -> router skips this mount



In [17]:
for pattern in ["a**b", "***", "/data/", "//x", "src/*.py"]:
    print(f"{pattern!r:12} -> {glob_defect(pattern)!r}")

'a**b'       -> "'**' inside a component ('a**b') — use '**' as a whole path segment"
'***'        -> "'**' inside a component ('***') — use '**' as a whole path segment"
'/data/'     -> "empty component — every '/' must separate non-empty segments"
'//x'        -> "empty component — every '/' must separate non-empty segments"
'src/*.py'   -> None


## 4. Ancestor death — pattern shorter than the mount

If the pattern is fully consumed *before* the mount path runs out, it
matched some directory **above** the mount point. The empty candidate can't
consume the remaining segments (`if not candidate: continue`), so it dies:

In [18]:
trace("/src", "/src/util")   # matched /src itself -- an ancestor, not a child dispatch

pattern '/src'  vs  mount '/src/util'
  start          : [('src',)]
  after 'src'    : [()]
  after 'util'   : []
  residuals()    = frozenset()
                   # empty set: no residual survived -> router skips this mount



## 5. The full pipeline: scope root + pattern → composed → per-mount dispatch

Upstream of `residuals` sits `composed_pattern` (glob.py:160): a **name-arm**
pattern (no `/`) floats — it becomes `root/**/pattern`, the gitignore rule —
while a **path-arm** pattern anchors under the root via `effective_pattern`.
`dispatch_table` below mirrors what the router does per mount at
`base.py:1960`.

In [19]:
print(composed_pattern(Path("/src"), "*.py"))        # name-arm floats: /src/**/*.py
print(composed_pattern(Path("/src"), "util/*.py"))   # path-arm anchors: /src/util/*.py
print(effective_pattern(Path("/src"), "/app.py"))    # leading / means the root itself

/src/**/*.py
/src/util/*.py
/src/app.py


In [20]:
def dispatch_table(root: str, pattern: str, mounts: list[str]) -> None:
    composed = composed_pattern(Path(root), pattern)
    print(f"scope root {root!r} + pattern {pattern!r}  =>  composed {composed!r}")
    for mount in mounts:
        parts = residuals(composed, Path(mount))
        print(f"  {mount:14} residuals() = {parts}")
        local = sorted(render_residual(c) for c in parts if c)  # base.py:1960 verbatim
        if local:
            print(f"  {'':14} dispatched with {local}")
        if any(not c for c in parts):
            print(f"  {'':14} # () bind point: parent backend serves that row")
        if not parts:
            print(f"  {'':14} # empty set: mount not dispatched")

dispatch_table("/src", "**/*.py", ["/src/util", "/src/vendor", "/docs", "/src/util/deep"])

scope root '/src' + pattern '**/*.py'  =>  composed '/src/**/*.py'
  /src/util      residuals() = frozenset({('**', '*.py')})
                 dispatched with ['/**/*.py']
  /src/vendor    residuals() = frozenset({('**', '*.py')})
                 dispatched with ['/**/*.py']
  /docs          residuals() = frozenset()
                 # empty set: mount not dispatched
  /src/util/deep residuals() = frozenset({('**', '*.py')})
                 dispatched with ['/**/*.py']


## 6. The other channel — rows already in hand (ADR 034)

Residuation serves the **scope** channel (`paths=`): searching *under*
roots across mounts. Rows a caller already holds never take that road —
chained `glob(observations=...)` is a pure predicate over paths in hand:
`filter_paths` (glob.py:99). No storage, no residuals, duplicates and order
preserved.

In [12]:
held = [Path("/src/app.py"), Path("/src/util/io.py"), Path("/docs/notes.md"), Path("/src/util/io.py")]
filter_paths(held, "src/**/*.py")

['/src/app.py', '/src/util/io.py', '/src/util/io.py']

## PLAY — exercises

1. **Predict, then run**: write down the frontier after each segment for
   `trace("/a/**/a/*.md", "/a/a/a")` — then run it. Did you keep the fork
   alive on the second `a`?
2. **Widen the frontier**: craft a pattern + mount that ends with exactly
   **four** live residuals. (Hint: section 2 got three from two `**/literal`
   pairs — what does a third pair do?)
3. **Read the router**: `trace("/src/*", "/src/util")` ends at the bind
   point. Find the line in `base.py` (~1957) that drops it from child
   dispatch, and say in one sentence why the parent backend serves that row.
4. **Wired to live work**: run
   `dispatch_table("/", "*.py", [<every mount above>])` — confirm the
   name-arm float means *no* mount is ever dead for a slash-free pattern.
   That's why `composed_pattern`, not the raw pattern, is what the router
   probes with (`base.py:1967`).